In [1]:
# ============================================================
# Notebook 27
# 27_oracle_probability_positive_control.ipynb
#
# Purpose:
#   Synthetic/oracle probability positive-control diagnostic.
#
# Reviewer issue addressed:
#   The main manuscript contains many negative controls. This notebook
#   tests whether the AURORA-compatible diagnostic allocation framework
#   can express timing information when probability signals are made
#   deliberately informative using oracle-with-noise probabilities.
#
# Core design:
#   p_tilde = (1 - kappa) * p_prior + kappa * one_hot(y_t)
#
#   For each kappa, construct:
#     1. Ordered oracle probabilities
#     2. Shuffled oracle probabilities
#     3. No-probability constant-lambda control
#
# Important:
#   This is deliberately non-deployable. It uses realized labels to build
#   synthetic oracle probabilities. The purpose is diagnostic only, not
#   investment performance estimation.
#
# Main outputs:
#   outputs/AURORA_TWETF/oracle_probability_positive_control/run_<RUN_ID>/
#     tables/table_S35_oracle_positive_control_design.csv
#     tables/table_S36_oracle_positive_control_performance.csv
#     tables/table_S37_oracle_ordered_vs_shuffled_comparison.csv
#     returns/oracle_positive_control_returns.parquet
#     weights/oracle_positive_control_weights.parquet
#     diagnostics/oracle_positive_control_diagnostics.csv
#     reports/NOTEBOOK27_oracle_positive_control_validation_report.json
#
# Research diagnostics only. Not financial advice.
# ============================================================

from __future__ import annotations

import json
import math
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Google Drive mount skipped or already mounted:", repr(e))

import numpy as np
import pandas as pd

try:
    from scipy.optimize import minimize
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False
    print("scipy.optimize unavailable. Optimizer will use fallback allocations.")

# ============================================================
# 0. Paths and global settings
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")
OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE

DATA_ROOT = PUBLICATION_ROOT / "data"
MODELING_DIR = DATA_ROOT / "modeling"
PANEL_DIR = DATA_ROOT / "panels"
RAW_YF_DIR = DATA_ROOT / "raw_yfinance"

TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
FIGURE_DIR = OUTPUT_ROOT / "figures"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "oracle_probability_positive_control" / f"run_{RUN_ID}"
TABLE_RUN_DIR = RUN_ROOT / "tables"
REPORT_RUN_DIR = RUN_ROOT / "reports"
DIAG_DIR = RUN_ROOT / "diagnostics"
RETURN_DIR = RUN_ROOT / "returns"
WEIGHT_DIR = RUN_ROOT / "weights"

for d in [
    RUN_ROOT,
    TABLE_RUN_DIR,
    REPORT_RUN_DIR,
    DIAG_DIR,
    RETURN_DIR,
    WEIGHT_DIR,
    TABLE_DIR,
    REPORT_DIR,
    FIGURE_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# Evaluation window
EVAL_START = pd.Timestamp("2024-11-27")
EVAL_END = pd.Timestamp("2026-03-25")

# Pre-evaluation window used only for class-prior estimation.
# The oracle component itself deliberately uses evaluation labels.
PRIOR_END = EVAL_START - pd.Timedelta(days=1)

# Dataset / target settings
ETF_UNIVERSE = ["0050", "006208", "00692", "00881"]
CASH_COL = "cash"
ASSET_COLS = ETF_UNIVERSE + [CASH_COL]
CLASS_LABELS = [0, 1, 2, 3, 4]
TARGET_20D = "TAIEX_regime_fixed_20d"
TARGET_60D = "TAIEX_regime_fixed_60d"

# AURORA10-UAMV-B-like diagnostic allocation settings
ANNUALIZATION_DAYS = 252
TRANSACTION_COST_BPS = 10
TRANSACTION_COST_RATE = TRANSACTION_COST_BPS / 10000.0
REBALANCE_CONVENTION = "month_end"

ALPHA_20 = 0.30
ALPHA_60 = 0.70
LAMBDA0 = 10.0
GAMMA_U = 2.5
GAMMA_B = 2.0
RHO = 0.25

MU_LOOKBACK = 63
COV_LOOKBACK = 126
MEAN_SHRINKAGE = 0.60
MOMENTUM_WEIGHT = 0.40
REGIME_TILT_STRENGTH = 0.25

GENERAL_ETF_CAP = 0.45
ETF_00881_CAP = 0.30
CASH_CAP = 0.60

# Positive-control signal strengths
KAPPA_VALUES = [0.00, 0.10, 0.25, 0.50, 0.75, 1.00]

# Shuffling seeds
BASE_RANDOM_SEED = 20260727
N_SHUFFLE_REPLICATIONS = 5

# Constant-lambda / no-probability control
CONSTANT_LAMBDA_CONTROL = 34.0

print("=" * 100)
print("AURORA-TWETF Notebook 27: Oracle probability positive control")
print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", RUN_ROOT)
print("Evaluation window:", EVAL_START.date(), "to", EVAL_END.date())
print("=" * 100)

# ============================================================
# 1. Utilities
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    path = Path(path)
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []
    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })
    return pd.DataFrame(rows)

def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def read_table_auto(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path, low_memory=False)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df[df["date"].notna()].copy()
        df = df.set_index("date")
    elif "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df[df["Date"].notna()].copy()
        df = df.set_index("Date")
    else:
        try:
            idx = pd.to_datetime(df.index, errors="coerce")
            if pd.Series(idx).notna().mean() > 0.50:
                df.index = idx
        except Exception:
            pass

    df.index.name = "date"
    return df.sort_index()

def normalize_name(x):
    return "".join(ch for ch in str(x).lower() if ch.isalnum())

def write_table(df, filename_stem, index=False, also_global=True):
    local_path = TABLE_RUN_DIR / f"{filename_stem}.csv"
    df.to_csv(local_path, index=index)
    print("Saved:", local_path)

    global_path = None
    if also_global:
        global_path = TABLE_DIR / f"{filename_stem}.csv"
        df.to_csv(global_path, index=index)
        print("Saved:", global_path)

    return local_path, global_path

def write_rounded_table(df, filename_stem, digits=6, also_global=True):
    out = df.copy()
    for c in out.select_dtypes(include=[np.number]).columns:
        out[c] = out[c].round(digits)
    return write_table(out, filename_stem, index=False, also_global=also_global)

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
        .replace(".", "_")
        .replace("%", "pct")
        .replace("-", "_")
        .replace("+", "plus")
        .replace("=", "_")
    )

# ============================================================
# 2. Load modeling labels and ETF returns
# ============================================================

print("\n" + "=" * 100)
print("Loading modeling labels and ETF returns")
print("=" * 100)

LABEL_FILE_CANDIDATES = [
    MODELING_DIR / "AURORA_TWETF_features_with_labels.parquet",
    MODELING_DIR / "AURORA_TWETF_features_with_labels.csv",
    MODELING_DIR / "features_with_labels.parquet",
    MODELING_DIR / "features_with_labels.csv",
]

label_path = find_first_existing(LABEL_FILE_CANDIDATES)
if label_path is None:
    raise FileNotFoundError(
        "Could not find modeling label file. Tried:\n"
        + "\n".join(str(p) for p in LABEL_FILE_CANDIDATES)
    )

labels_df = read_table_auto(label_path)

for target in [TARGET_20D, TARGET_60D]:
    if target not in labels_df.columns:
        raise ValueError(f"Missing target column {target} in {label_path}. Available columns include: {list(labels_df.columns)[:40]}")

labels_df = labels_df[[TARGET_20D, TARGET_60D]].dropna().copy()
labels_df[TARGET_20D] = labels_df[TARGET_20D].astype(int)
labels_df[TARGET_60D] = labels_df[TARGET_60D].astype(int)

print("Label file:", label_path)
print("Label shape:", labels_df.shape)
print("Label dates:", labels_df.index.min().date(), "to", labels_df.index.max().date())

ETF_RETURN_PANEL_CANDIDATES = [
    PANEL_DIR / "AURORA_etf_return_panel.parquet",
    PANEL_DIR / "AURORA_TWETF_etf_return_panel.parquet",
    PANEL_DIR / "etf_return_panel.parquet",
    PANEL_DIR / "AURORA_etf_returns.parquet",
    PANEL_DIR / "AURORA_etf_return_panel.csv",
]

etf_return_path = find_first_existing(ETF_RETURN_PANEL_CANDIDATES)

def load_etf_returns_from_panel(path):
    df = read_table_auto(path)
    rename = {}
    for c in df.columns:
        nc = normalize_name(c)
        if "006208" in nc:
            rename[c] = "006208"
        elif "00692" in nc:
            rename[c] = "00692"
        elif "00881" in nc:
            rename[c] = "00881"
        elif "0050" in nc:
            rename[c] = "0050"
    df = df.rename(columns=rename)
    missing = [c for c in ETF_UNIVERSE if c not in df.columns]
    if missing:
        raise ValueError(f"ETF return panel missing columns: {missing}")
    out = df[ETF_UNIVERSE].apply(pd.to_numeric, errors="coerce")
    out = out.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return out.sort_index()

def load_raw_price_for_etf(symbol):
    candidates = [
        RAW_YF_DIR / f"{symbol}_{symbol}_TW.csv",
        RAW_YF_DIR / f"{symbol}.TW.csv",
        RAW_YF_DIR / f"{symbol}.csv",
    ]
    p = find_first_existing(candidates)
    if p is None:
        raise FileNotFoundError(f"Could not find raw yfinance file for {symbol}. Tried: {candidates}")

    df = read_table_auto(p)
    desired_adj = f"Adj Close_{symbol}.TW"
    desired_close = f"Close_{symbol}.TW"

    if desired_adj in df.columns:
        col = desired_adj
    elif desired_close in df.columns:
        col = desired_close
    else:
        adj_cols = [c for c in df.columns if "adjclose" in normalize_name(c)]
        close_cols = [c for c in df.columns if "close" in normalize_name(c) and "adj" not in normalize_name(c)]
        if adj_cols:
            col = adj_cols[0]
        elif close_cols:
            col = close_cols[0]
        else:
            raise ValueError(f"No close or adjusted-close column found for {symbol} in {p}. Columns: {list(df.columns)[:20]}")

    s = pd.to_numeric(df[col], errors="coerce").dropna()
    s.name = symbol
    return s.sort_index()

def load_etf_returns_from_raw():
    prices = []
    for sym in ETF_UNIVERSE:
        prices.append(load_raw_price_for_etf(sym))
    price_df = pd.concat(prices, axis=1).sort_index()
    ret_df = price_df.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how="all").fillna(0.0)
    return ret_df[ETF_UNIVERSE].copy()

if etf_return_path is not None:
    etf_returns = load_etf_returns_from_panel(etf_return_path)
    print("ETF return panel:", etf_return_path)
else:
    etf_returns = load_etf_returns_from_raw()
    etf_return_path = "raw_yfinance_reconstructed"
    print("ETF returns reconstructed from raw_yfinance files")

print("ETF return shape:", etf_returns.shape)
print("ETF return dates:", etf_returns.index.min().date(), "to", etf_returns.index.max().date())

# Evaluation label-return common dates
eval_dates = (
    labels_df.index
    .intersection(etf_returns.index)
    .sort_values()
)
eval_dates = eval_dates[(eval_dates >= EVAL_START) & (eval_dates <= EVAL_END)]

if len(eval_dates) == 0:
    raise ValueError("No evaluation dates after aligning labels and ETF returns.")

print("Aligned evaluation dates:", len(eval_dates), eval_dates.min().date(), "to", eval_dates.max().date())

# Prior estimation dates
prior_dates = labels_df.index[labels_df.index < EVAL_START].sort_values()
if len(prior_dates) < 50:
    raise ValueError(f"Too few prior dates for class prior estimation: {len(prior_dates)}")

# ============================================================
# 3. Probability construction and scoring
# ============================================================

def class_prior_from_labels(y):
    y = pd.Series(y).dropna().astype(int)
    counts = y.value_counts().reindex(CLASS_LABELS).fillna(0.0).astype(float)
    if counts.sum() <= 0:
        return np.ones(len(CLASS_LABELS)) / len(CLASS_LABELS)
    return (counts / counts.sum()).values

prior20 = class_prior_from_labels(labels_df.loc[prior_dates, TARGET_20D])
prior60 = class_prior_from_labels(labels_df.loc[prior_dates, TARGET_60D])

print("Prior 20d:", dict(zip(CLASS_LABELS, prior20.round(4))))
print("Prior 60d:", dict(zip(CLASS_LABELS, prior60.round(4))))

def onehot(labels):
    labels = pd.Series(labels).astype(int).values
    out = np.zeros((len(labels), len(CLASS_LABELS)), dtype=float)
    for i, y in enumerate(labels):
        if y in CLASS_LABELS:
            out[i, CLASS_LABELS.index(int(y))] = 1.0
        else:
            out[i, :] = 1.0 / len(CLASS_LABELS)
    return out

def normalize_proba_array(p):
    p = np.asarray(p, dtype=float)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p[p < 0] = 0.0
    row_sum = p.sum(axis=1, keepdims=True)
    bad = row_sum[:, 0] <= 0
    if bad.any():
        p[bad, :] = 1.0 / p.shape[1]
        row_sum = p.sum(axis=1, keepdims=True)
    return p / row_sum

def construct_oracle_probability(labels, prior_vec, kappa):
    oh = onehot(labels)
    prior = np.asarray(prior_vec, dtype=float).reshape(1, -1)
    p = (1.0 - float(kappa)) * prior + float(kappa) * oh
    return normalize_proba_array(p)

def shuffle_probability_rows(p, seed):
    rng = np.random.default_rng(seed)
    idx = np.arange(p.shape[0])
    rng.shuffle(idx)
    return p[idx, :]

def probability_features_from_p20_p60(index, p20, p60, alpha20=ALPHA_20, alpha60=ALPHA_60):
    p20 = normalize_proba_array(p20)
    p60 = normalize_proba_array(p60)

    s = alpha20 + alpha60
    if s <= 0:
        alpha20, alpha60 = 0.5, 0.5
    else:
        alpha20, alpha60 = alpha20 / s, alpha60 / s

    p = alpha20 * p20 + alpha60 * p60
    p = normalize_proba_array(p)

    clipped = np.clip(p, 1e-12, 1.0)
    entropy = -np.sum(clipped * np.log(clipped), axis=1)
    normalized_entropy = entropy / np.log(len(CLASS_LABELS))

    class_values = np.asarray(CLASS_LABELS, dtype=float)
    expected_class = p @ class_values
    ordinal_variance = p @ (class_values ** 2) - expected_class ** 2

    out = pd.DataFrame(index=pd.DatetimeIndex(index).sort_values())
    out.index.name = "date"
    for i, cls in enumerate(CLASS_LABELS):
        out[f"proba_class_{cls}"] = p[:, i]
    out["expected_class"] = expected_class
    out["normalized_entropy"] = normalized_entropy
    out["entropy"] = entropy
    out["confidence_score"] = 1.0 - normalized_entropy
    out["ordinal_variance"] = ordinal_variance
    out["p_bearish"] = p[:, 0] + p[:, 1]
    out["p_bullish"] = p[:, 3] + p[:, 4]
    out["risk_on_score"] = (expected_class - 2.0) / 2.0
    out["lambda_dynamic"] = LAMBDA0 * (1.0 + GAMMA_U * out["normalized_entropy"] + GAMMA_B * out["p_bearish"])
    return out

def multiclass_brier(y_true, p):
    y_true = pd.Series(y_true).astype(int).values
    p = normalize_proba_array(p)
    oh = onehot(y_true)
    return float(np.mean(np.sum((p - oh) ** 2, axis=1)))

def hard_accuracy(y_true, p):
    y_true = pd.Series(y_true).astype(int).values
    pred = np.asarray(CLASS_LABELS)[np.argmax(normalize_proba_array(p), axis=1)]
    return float(np.mean(pred == y_true))

def probability_summary(y20, y60, p20, p60, features):
    return {
        "brier_20d": multiclass_brier(y20, p20),
        "brier_60d": multiclass_brier(y60, p60),
        "weighted_brier": ALPHA_20 * multiclass_brier(y20, p20) + ALPHA_60 * multiclass_brier(y60, p60),
        "hard_accuracy_20d": hard_accuracy(y20, p20),
        "hard_accuracy_60d": hard_accuracy(y60, p60),
        "avg_entropy": float(features["normalized_entropy"].mean()),
        "avg_bearish_probability": float(features["p_bearish"].mean()),
        "avg_dynamic_lambda": float(features["lambda_dynamic"].mean()),
        "median_dynamic_lambda": float(features["lambda_dynamic"].median()),
    }

# ============================================================
# 4. AURORA-compatible optimizer and backtest
# ============================================================

def make_rebalance_dates(index):
    index = pd.DatetimeIndex(index).sort_values()
    ser = pd.Series(index=index, data=index)
    if REBALANCE_CONVENTION == "month_start":
        return pd.DatetimeIndex(ser.groupby(index.to_period("M")).min().values).sort_values()
    return pd.DatetimeIndex(ser.groupby(index.to_period("M")).max().values).sort_values()

def cap_vector():
    return np.array([GENERAL_ETF_CAP, GENERAL_ETF_CAP, GENERAL_ETF_CAP, ETF_00881_CAP, CASH_CAP], dtype=float)

def cap_and_normalize_weights(w):
    w = np.asarray(w, dtype=float).copy()
    caps = cap_vector()

    w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
    w = np.maximum(w, 0.0)
    w = np.minimum(w, caps)

    for _ in range(50):
        gap = 1.0 - w.sum()
        if abs(gap) < 1e-10:
            break

        if gap > 0:
            capacity = caps - w
            capacity[capacity < 0] = 0.0
            if capacity.sum() <= 1e-12:
                break
            w += gap * capacity / capacity.sum()
            w = np.minimum(w, caps)
        else:
            positive = w > 0
            if positive.sum() == 0:
                break
            w[positive] += gap * w[positive] / w[positive].sum()
            w = np.maximum(w, 0.0)

    if abs(w.sum() - 1.0) > 1e-6:
        # Conservative feasible fallback
        w = np.array([0.10, 0.10, 0.10, 0.10, 0.60], dtype=float)

    return w / w.sum()

def initial_weight():
    return np.array([0.10, 0.10, 0.10, 0.10, 0.60], dtype=float)

def regime_tilt_vector(features_row):
    risk_on = float(features_row.get("risk_on_score", 0.0))
    p_bearish = float(features_row.get("p_bearish", 0.0))
    p_bullish = float(features_row.get("p_bullish", 0.0))

    tilt = pd.Series(0.0, index=ETF_UNIVERSE)
    tilt["0050"] += 0.15 * risk_on
    tilt["006208"] += 0.15 * risk_on
    tilt["00692"] += 0.05 * risk_on + 0.10 * p_bearish
    tilt["00881"] += 0.35 * risk_on + 0.15 * p_bullish - 0.20 * p_bearish
    return tilt.values

def estimate_moments_for_date(etf_returns, dt, features_row, use_regime_tilt=True):
    hist = etf_returns.loc[etf_returns.index < dt, ETF_UNIVERSE].dropna(how="all").fillna(0.0)

    if len(hist) < max(MU_LOOKBACK, COV_LOOKBACK):
        return None, None

    r_mu = hist.tail(MU_LOOKBACK)
    r_cov = hist.tail(COV_LOOKBACK)

    mean_daily = r_mu.mean().values
    cumulative = (1.0 + r_mu).prod().values - 1.0
    momentum_daily = cumulative / max(len(r_mu), 1)

    risky_mu = (1.0 - MEAN_SHRINKAGE) * (
        (1.0 - MOMENTUM_WEIGHT) * mean_daily
        + MOMENTUM_WEIGHT * momentum_daily
    )

    if use_regime_tilt:
        realized_vol = r_cov.std().replace(0.0, np.nan)
        vol_scale = realized_vol.median()
        if not np.isfinite(vol_scale) or vol_scale <= 0:
            vol_scale = 0.01
        risky_mu = risky_mu + REGIME_TILT_STRENGTH * regime_tilt_vector(features_row) * vol_scale / ANNUALIZATION_DAYS

    cov = r_cov.cov().values
    cov = np.nan_to_num(cov, nan=0.0, posinf=0.0, neginf=0.0)

    avg_var = np.nanmean(np.diag(cov))
    if not np.isfinite(avg_var) or avg_var <= 0:
        avg_var = 1e-6

    cov = cov + np.eye(len(ETF_UNIVERSE)) * max(1e-10, avg_var * 0.10)

    mu = np.concatenate([risky_mu, [0.0]])
    sigma = np.zeros((5, 5), dtype=float)
    sigma[:4, :4] = cov
    sigma[4, 4] = 1e-12

    return mu, sigma

def optimize_weight(mu, sigma, lam, prev_w):
    prev_w = cap_and_normalize_weights(prev_w)
    caps = cap_vector()
    bounds = [(0.0, caps[i]) for i in range(5)]
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

    if mu is None or sigma is None or not SCIPY_AVAILABLE:
        return prev_w, {
            "solver_success": False,
            "fallback_used": True,
            "message": "missing_moments_or_scipy",
            "objective_value": np.nan,
            "equality_residual": abs(prev_w.sum() - 1.0),
            "max_bound_violation": 0.0,
        }

    def objective(w):
        w = np.asarray(w, dtype=float)
        return -(
            float(mu @ w)
            - float(lam) * float(w.T @ sigma @ w)
            - float(RHO) * float(np.sum((w - prev_w) ** 2))
        )

    x0 = prev_w.copy()

    try:
        res = minimize(
            objective,
            x0=x0,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 300, "ftol": 1e-10, "disp": False},
        )

        if res.success and np.all(np.isfinite(res.x)):
            w = cap_and_normalize_weights(res.x)
            success = True
            fallback = False
            msg = str(res.message)
            obj = -float(res.fun)
        else:
            w = prev_w.copy()
            success = False
            fallback = True
            msg = str(getattr(res, "message", "optimizer_failed"))
            obj = np.nan
    except Exception as e:
        w = prev_w.copy()
        success = False
        fallback = True
        msg = repr(e)[:160]
        obj = np.nan

    eq_resid = abs(float(w.sum() - 1.0))
    max_bound_viol = float(max(0.0, np.max(w - caps), -np.min(w)))

    return w, {
        "solver_success": bool(success),
        "fallback_used": bool(fallback),
        "message": msg,
        "objective_value": obj,
        "equality_residual": eq_resid,
        "max_bound_violation": max_bound_viol,
    }

def performance_metrics_from_returns(r):
    r = pd.Series(r).dropna().astype(float)
    n = len(r)

    if n == 0:
        return {}

    equity = (1.0 + r).cumprod()
    dd = equity / equity.cummax() - 1.0

    total_return = float(equity.iloc[-1] - 1.0)
    annual_return = float(equity.iloc[-1] ** (ANNUALIZATION_DAYS / max(n, 1)) - 1.0)

    daily_vol = float(r.std(ddof=1)) if n > 1 else np.nan
    annual_vol = float(daily_vol * np.sqrt(ANNUALIZATION_DAYS)) if np.isfinite(daily_vol) else np.nan
    sharpe = float((r.mean() / daily_vol) * np.sqrt(ANNUALIZATION_DAYS)) if np.isfinite(daily_vol) and daily_vol > 0 else np.nan

    downside = r[r < 0]
    downside_vol = float(downside.std(ddof=1)) if len(downside) > 1 else np.nan
    sortino = float((r.mean() / downside_vol) * np.sqrt(ANNUALIZATION_DAYS)) if np.isfinite(downside_vol) and downside_vol > 0 else np.nan

    max_dd = float(dd.min())
    calmar = float(annual_return / abs(max_dd)) if max_dd < 0 else np.nan

    return {
        "n_days": int(n),
        "start_date": str(r.index.min().date()),
        "end_date": str(r.index.max().date()),
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": max_dd,
        "calmar": calmar,
        "avg_daily_return": float(r.mean()),
        "daily_volatility": daily_vol,
        "hit_rate": float((r > 0).mean()),
        "final_equity": float(equity.iloc[-1]),
    }

def run_aurora_like_backtest(
    policy_name,
    etf_returns,
    features,
    mode="dynamic",
    constant_lambda=None,
    use_regime_tilt=True,
):
    dates = pd.DatetimeIndex(features.index).intersection(etf_returns.index).sort_values()
    dates = dates[(dates >= EVAL_START) & (dates <= EVAL_END)]

    rebalance_dates = set(make_rebalance_dates(dates))
    current_w = initial_weight()

    ret_rows = []
    weight_rows = []
    diag_rows = []

    for dt in dates:
        tc = 0.0
        is_rebalance = dt in rebalance_dates

        frow = features.loc[dt]

        if is_rebalance:
            mu, sigma = estimate_moments_for_date(
                etf_returns=etf_returns,
                dt=dt,
                features_row=frow,
                use_regime_tilt=use_regime_tilt,
            )

            if mode == "dynamic":
                lam = float(frow["lambda_dynamic"])
            elif mode == "constant":
                lam = float(constant_lambda)
            else:
                raise ValueError(f"Unknown mode: {mode}")

            new_w, opt_diag = optimize_weight(mu, sigma, lam, current_w)

            turnover = float(np.sum(np.abs(new_w - current_w)))
            tc = turnover * TRANSACTION_COST_RATE
            current_w = new_w.copy()

            diag_rows.append({
                "date": dt,
                "policy_name": policy_name,
                "mode": mode,
                "lambda_value": lam,
                "is_rebalance": True,
                "turnover": turnover,
                "transaction_cost": tc,
                "cash_weight": float(current_w[4]),
                "at_cash_cap": float(current_w[4] >= CASH_CAP - 1e-6),
                "avg_equity_exposure": float(current_w[:4].sum()),
                "expected_class": float(frow["expected_class"]),
                "normalized_entropy": float(frow["normalized_entropy"]),
                "p_bearish": float(frow["p_bearish"]),
                **opt_diag,
            })

        asset_ret = etf_returns.loc[dt, ETF_UNIVERSE].fillna(0.0).values
        gross_return = float(np.dot(current_w[:4], asset_ret))
        net_return = gross_return - tc

        ret_rows.append({
            "date": dt,
            "policy_name": policy_name,
            "gross_return": gross_return,
            "transaction_cost": tc,
            "net_return": net_return,
        })

        weight_rows.append({
            "date": dt,
            "policy_name": policy_name,
            "0050": float(current_w[0]),
            "006208": float(current_w[1]),
            "00692": float(current_w[2]),
            "00881": float(current_w[3]),
            "cash": float(current_w[4]),
            "is_rebalance": bool(is_rebalance),
        })

        # Drift weights after returns, with cash return zero.
        post_values = current_w.copy()
        post_values[:4] = post_values[:4] * (1.0 + asset_ret)
        post_values[4] = post_values[4]
        if post_values.sum() > 0:
            current_w = post_values / post_values.sum()
        else:
            current_w = initial_weight()

    returns = pd.DataFrame(ret_rows).set_index("date").sort_index()
    weights = pd.DataFrame(weight_rows).set_index("date").sort_index()
    diagnostics = pd.DataFrame(diag_rows).set_index("date").sort_index() if diag_rows else pd.DataFrame()

    returns["equity"] = (1.0 + returns["net_return"]).cumprod()
    returns["drawdown"] = returns["equity"] / returns["equity"].cummax() - 1.0

    return returns, weights, diagnostics

# ============================================================
# 5. Build and run oracle positive-control experiments
# ============================================================

print("\n" + "=" * 100)
print("Running oracle positive-control experiments")
print("=" * 100)

y20_eval = labels_df.loc[eval_dates, TARGET_20D].astype(int)
y60_eval = labels_df.loc[eval_dates, TARGET_60D].astype(int)

all_returns = []
all_weights = []
all_diagnostics = []
perf_rows = []
design_rows = []
ordered_vs_shuffled_rows = []

# Constant no-probability features: class-prior blended probabilities for all dates.
p20_prior_eval = np.tile(prior20.reshape(1, -1), (len(eval_dates), 1))
p60_prior_eval = np.tile(prior60.reshape(1, -1), (len(eval_dates), 1))
prior_features = probability_features_from_p20_p60(eval_dates, p20_prior_eval, p60_prior_eval)

const_returns, const_weights, const_diag = run_aurora_like_backtest(
    policy_name=f"constant_lambda_no_probability_lambda_{CONSTANT_LAMBDA_CONTROL:g}",
    etf_returns=etf_returns,
    features=prior_features,
    mode="constant",
    constant_lambda=CONSTANT_LAMBDA_CONTROL,
    use_regime_tilt=False,
)

all_returns.append(const_returns)
all_weights.append(const_weights)
all_diagnostics.append(const_diag)

const_perf = performance_metrics_from_returns(const_returns["net_return"])
const_perf.update({
    "policy_name": f"constant_lambda_no_probability_lambda_{CONSTANT_LAMBDA_CONTROL:g}",
    "control_type": "constant_lambda_no_probability",
    "kappa": np.nan,
    "shuffle_id": np.nan,
    "probability_order": "none",
    "weighted_brier": np.nan,
    "avg_entropy": float(prior_features["normalized_entropy"].mean()),
    "avg_bearish_probability": float(prior_features["p_bearish"].mean()),
    "avg_dynamic_lambda": float(prior_features["lambda_dynamic"].mean()),
    "avg_cash": float(const_weights["cash"].mean()),
    "cash_cap_binding_frequency": float((const_weights["cash"] >= CASH_CAP - 1e-6).mean()),
    "avg_equity_exposure": float(const_weights[ETF_UNIVERSE].sum(axis=1).mean()),
    "total_turnover": float(const_diag["turnover"].sum()) if not const_diag.empty and "turnover" in const_diag.columns else np.nan,
    "rebalance_count": int(len(const_diag)),
})
perf_rows.append(const_perf)

for kappa in KAPPA_VALUES:
    p20_ordered = construct_oracle_probability(y20_eval, prior20, kappa)
    p60_ordered = construct_oracle_probability(y60_eval, prior60, kappa)
    features_ordered = probability_features_from_p20_p60(eval_dates, p20_ordered, p60_ordered)

    psum = probability_summary(y20_eval, y60_eval, p20_ordered, p60_ordered, features_ordered)

    design_rows.append({
        "kappa": float(kappa),
        "probability_order": "ordered",
        "description": "(1-kappa)*pre-evaluation class prior + kappa*realized label one-hot vector",
        "deployability": "non-deployable_oracle_positive_control",
        **psum,
    })

    policy_name = f"oracle_ordered_kappa_{safe_name(kappa)}"

    ret_df, w_df, d_df = run_aurora_like_backtest(
        policy_name=policy_name,
        etf_returns=etf_returns,
        features=features_ordered,
        mode="dynamic",
        constant_lambda=None,
        use_regime_tilt=True,
    )

    all_returns.append(ret_df)
    all_weights.append(w_df)
    all_diagnostics.append(d_df)

    perf = performance_metrics_from_returns(ret_df["net_return"])
    perf.update({
        "policy_name": policy_name,
        "control_type": "ordered_oracle_probability",
        "kappa": float(kappa),
        "shuffle_id": np.nan,
        "probability_order": "ordered",
        **psum,
        "avg_cash": float(w_df["cash"].mean()),
        "cash_cap_binding_frequency": float((w_df["cash"] >= CASH_CAP - 1e-6).mean()),
        "avg_equity_exposure": float(w_df[ETF_UNIVERSE].sum(axis=1).mean()),
        "total_turnover": float(d_df["turnover"].sum()) if not d_df.empty and "turnover" in d_df.columns else np.nan,
        "rebalance_count": int(len(d_df)),
    })
    perf_rows.append(perf)

    # Shuffled controls: preserve marginal distribution of oracle probability vectors but destroy date ordering.
    shuffle_perf_rows = []

    for shuffle_id in range(N_SHUFFLE_REPLICATIONS):
        seed = BASE_RANDOM_SEED + int(kappa * 1000) + shuffle_id
        p20_shuf = shuffle_probability_rows(p20_ordered, seed=seed)
        p60_shuf = shuffle_probability_rows(p60_ordered, seed=seed + 9999)
        features_shuf = probability_features_from_p20_p60(eval_dates, p20_shuf, p60_shuf)

        psum_shuf = probability_summary(y20_eval, y60_eval, p20_shuf, p60_shuf, features_shuf)

        design_rows.append({
            "kappa": float(kappa),
            "probability_order": "shuffled",
            "shuffle_id": int(shuffle_id),
            "description": "Date order of oracle probability vectors is shuffled; marginal probability distribution is preserved.",
            "deployability": "non-deployable_shuffled_oracle_control",
            **psum_shuf,
        })

        shuf_policy = f"oracle_shuffled_kappa_{safe_name(kappa)}_seed_{shuffle_id}"

        shuf_ret, shuf_w, shuf_d = run_aurora_like_backtest(
            policy_name=shuf_policy,
            etf_returns=etf_returns,
            features=features_shuf,
            mode="dynamic",
            constant_lambda=None,
            use_regime_tilt=True,
        )

        all_returns.append(shuf_ret)
        all_weights.append(shuf_w)
        all_diagnostics.append(shuf_d)

        shuf_perf = performance_metrics_from_returns(shuf_ret["net_return"])
        shuf_perf.update({
            "policy_name": shuf_policy,
            "control_type": "shuffled_oracle_probability",
            "kappa": float(kappa),
            "shuffle_id": int(shuffle_id),
            "probability_order": "shuffled",
            **psum_shuf,
            "avg_cash": float(shuf_w["cash"].mean()),
            "cash_cap_binding_frequency": float((shuf_w["cash"] >= CASH_CAP - 1e-6).mean()),
            "avg_equity_exposure": float(shuf_w[ETF_UNIVERSE].sum(axis=1).mean()),
            "total_turnover": float(shuf_d["turnover"].sum()) if not shuf_d.empty and "turnover" in shuf_d.columns else np.nan,
            "rebalance_count": int(len(shuf_d)),
        })
        perf_rows.append(shuf_perf)
        shuffle_perf_rows.append(shuf_perf)

    # Ordered vs shuffled mean comparison for each kappa.
    shuf_df = pd.DataFrame(shuffle_perf_rows)
    ordered_perf = perf.copy()

    ordered_vs_shuffled_rows.append({
        "kappa": float(kappa),
        "ordered_policy": policy_name,
        "n_shuffle_replications": int(N_SHUFFLE_REPLICATIONS),
        "ordered_total_return": ordered_perf["total_return"],
        "shuffled_mean_total_return": float(shuf_df["total_return"].mean()),
        "ordered_minus_shuffled_total_return": ordered_perf["total_return"] - float(shuf_df["total_return"].mean()),
        "ordered_sharpe": ordered_perf["sharpe"],
        "shuffled_mean_sharpe": float(shuf_df["sharpe"].mean()),
        "ordered_minus_shuffled_sharpe": ordered_perf["sharpe"] - float(shuf_df["sharpe"].mean()),
        "ordered_sortino": ordered_perf["sortino"],
        "shuffled_mean_sortino": float(shuf_df["sortino"].mean()),
        "ordered_minus_shuffled_sortino": ordered_perf["sortino"] - float(shuf_df["sortino"].mean()),
        "ordered_max_drawdown": ordered_perf["max_drawdown"],
        "shuffled_mean_max_drawdown": float(shuf_df["max_drawdown"].mean()),
        "ordered_minus_shuffled_max_drawdown": ordered_perf["max_drawdown"] - float(shuf_df["max_drawdown"].mean()),
        "ordered_avg_cash": ordered_perf["avg_cash"],
        "shuffled_mean_avg_cash": float(shuf_df["avg_cash"].mean()),
        "ordered_minus_shuffled_avg_cash": ordered_perf["avg_cash"] - float(shuf_df["avg_cash"].mean()),
        "ordered_cash_cap_binding": ordered_perf["cash_cap_binding_frequency"],
        "shuffled_mean_cash_cap_binding": float(shuf_df["cash_cap_binding_frequency"].mean()),
        "ordered_minus_shuffled_cash_cap_binding": ordered_perf["cash_cap_binding_frequency"] - float(shuf_df["cash_cap_binding_frequency"].mean()),
        "ordered_weighted_brier": ordered_perf["weighted_brier"],
        "shuffled_mean_weighted_brier": float(shuf_df["weighted_brier"].mean()),
        "ordered_minus_shuffled_weighted_brier": ordered_perf["weighted_brier"] - float(shuf_df["weighted_brier"].mean()),
    })

    print(
        f"kappa={kappa:.2f} | ordered TR={perf['total_return']:.4f}, "
        f"Sharpe={perf['sharpe']:.4f}, MDD={perf['max_drawdown']:.4f}, "
        f"avg cash={perf['avg_cash']:.4f}, cap={perf['cash_cap_binding_frequency']:.4f}"
    )

# ============================================================
# 6. Export combined outputs
# ============================================================

print("\n" + "=" * 100)
print("Exporting Notebook 27 outputs")
print("=" * 100)

performance_df = pd.DataFrame(perf_rows)
design_df = pd.DataFrame(design_rows)
ordered_vs_shuffled_df = pd.DataFrame(ordered_vs_shuffled_rows)

# Compact design table for Supplement Table S35
s35 = pd.DataFrame([
    {
        "component": "Positive-control probability",
        "specification": r"\(\tilde{\mathbf p}_t=(1-\kappa)\mathbf p_{\mathrm{prior}}+\kappa\mathbf e_{y_t}\)",
        "interpretation": "Deliberately non-deployable oracle-with-noise probability sequence used to test diagnostic power.",
    },
    {
        "component": "Prior probability",
        "specification": "Pre-evaluation empirical class-prior distribution estimated before the aligned evaluation window.",
        "interpretation": "Defines the no-timing baseline in the oracle mixture.",
    },
    {
        "component": "Oracle component",
        "specification": "One-hot realized 20-day and 60-day regime labels in the aligned evaluation window.",
        "interpretation": "Introduces controlled signal strength; not available in real time.",
    },
    {
        "component": "Kappa values",
        "specification": ", ".join([str(k) for k in KAPPA_VALUES]),
        "interpretation": "Signal strength grid from no oracle information to perfect realized-label information.",
    },
    {
        "component": "Ordered control",
        "specification": "Oracle probabilities kept on their original dates.",
        "interpretation": "Tests whether informative probabilities can affect the allocation path.",
    },
    {
        "component": "Shuffled control",
        "specification": f"Oracle probability rows shuffled across dates; {N_SHUFFLE_REPLICATIONS} replications per kappa.",
        "interpretation": "Preserves marginal probability distribution but destroys timing.",
    },
    {
        "component": "Constant-lambda control",
        "specification": f"No-probability constant-lambda control with lambda={CONSTANT_LAMBDA_CONTROL:g}.",
        "interpretation": "Tests behavior without dynamic probability timing.",
    },
    {
        "component": "Claim use",
        "specification": "Diagnostic positive control only.",
        "interpretation": "Demonstrates whether the attribution framework can detect timing information when such information is injected.",
    },
])

# Performance table for Supplement Table S36
s36_cols = [
    "policy_name",
    "control_type",
    "kappa",
    "shuffle_id",
    "probability_order",
    "n_days",
    "weighted_brier",
    "avg_entropy",
    "avg_bearish_probability",
    "avg_dynamic_lambda",
    "total_return",
    "sharpe",
    "sortino",
    "max_drawdown",
    "avg_cash",
    "cash_cap_binding_frequency",
    "avg_equity_exposure",
    "total_turnover",
]
s36 = performance_df[[c for c in s36_cols if c in performance_df.columns]].copy()

# Ordered-only compact performance table
s36_ordered_compact = s36[
    (s36["control_type"].isin(["ordered_oracle_probability", "constant_lambda_no_probability"]))
].copy()

# Ordered vs shuffled table for Supplement Table S37
s37 = ordered_vs_shuffled_df.copy()

write_table(s35, "table_S35_oracle_positive_control_design")
write_rounded_table(s36, "table_S36_oracle_positive_control_performance")
write_rounded_table(s36_ordered_compact, "table_S36b_oracle_positive_control_ordered_compact")
write_rounded_table(s37, "table_S37_oracle_ordered_vs_shuffled_comparison")

# Save full combined returns, weights, diagnostics
all_returns_df = pd.concat(
    [df.reset_index() for df in all_returns],
    axis=0,
    ignore_index=True,
)

all_weights_df = pd.concat(
    [df.reset_index() for df in all_weights],
    axis=0,
    ignore_index=True,
)

nonempty_diag = [df.reset_index() for df in all_diagnostics if df is not None and not df.empty]
all_diag_df = pd.concat(nonempty_diag, axis=0, ignore_index=True) if nonempty_diag else pd.DataFrame()

all_returns_df.to_parquet(RETURN_DIR / "oracle_positive_control_returns.parquet", index=False)
all_returns_df.to_csv(RETURN_DIR / "oracle_positive_control_returns.csv", index=False)

all_weights_df.to_parquet(WEIGHT_DIR / "oracle_positive_control_weights.parquet", index=False)
all_weights_df.to_csv(WEIGHT_DIR / "oracle_positive_control_weights.csv", index=False)

all_diag_df.to_csv(DIAG_DIR / "oracle_positive_control_diagnostics.csv", index=False)

design_df.to_csv(DIAG_DIR / "oracle_probability_design_full.csv", index=False)
performance_df.to_csv(DIAG_DIR / "oracle_positive_control_performance_full.csv", index=False)
ordered_vs_shuffled_df.to_csv(DIAG_DIR / "oracle_ordered_vs_shuffled_full.csv", index=False)

# Global copies of major machine-readable diagnostics
all_returns_df.to_csv(OUTPUT_ROOT / "diagnostics" / "oracle_positive_control_returns.csv", index=False)
all_weights_df.to_csv(OUTPUT_ROOT / "diagnostics" / "oracle_positive_control_weights.csv", index=False)
ordered_vs_shuffled_df.to_csv(OUTPUT_ROOT / "diagnostics" / "oracle_ordered_vs_shuffled_full.csv", index=False)

# ============================================================
# 7. Interpretation helper
# ============================================================

interpret_rows = []

# Compare whether ordered improves against shuffled as kappa increases
if not s37.empty:
    for _, row in s37.iterrows():
        k = row["kappa"]
        interpret_rows.append({
            "section": "Oracle positive control",
            "item": f"kappa={k}",
            "finding": (
                "Ordered oracle outperforms shuffled on Sharpe"
                if row["ordered_minus_shuffled_sharpe"] > 0
                else "Ordered oracle does not outperform shuffled on Sharpe"
            ),
            "evidence": (
                f"Ordered-minus-shuffled Sharpe={row['ordered_minus_shuffled_sharpe']:.4f}; "
                f"TR diff={row['ordered_minus_shuffled_total_return']:.4f}; "
                f"MDD diff={row['ordered_minus_shuffled_max_drawdown']:.4f}; "
                f"cash-cap diff={row['ordered_minus_shuffled_cash_cap_binding']:.4f}."
            ),
        })

# Diagnostic conclusion
ordered_rows = performance_df[performance_df["control_type"] == "ordered_oracle_probability"].copy()
if not ordered_rows.empty:
    corr_kappa_sharpe = ordered_rows[["kappa", "sharpe"]].corr().iloc[0, 1]
    corr_kappa_cash = ordered_rows[["kappa", "avg_cash"]].corr().iloc[0, 1]
    interpret_rows.append({
        "section": "Oracle positive control",
        "item": "kappa trend",
        "finding": "Signal-strength trend diagnostic",
        "evidence": (
            f"Correlation between kappa and ordered-oracle Sharpe={corr_kappa_sharpe:.4f}; "
            f"correlation between kappa and average cash={corr_kappa_cash:.4f}."
        ),
    })

interpret_df = pd.DataFrame(interpret_rows)
write_table(interpret_df, "notebook27_oracle_positive_control_interpretation_helper")

# ============================================================
# 8. Validation report and manifest
# ============================================================

validation = {
    "project_code": PROJECT_CODE,
    "notebook": "27_oracle_probability_positive_control",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "purpose": (
        "Synthetic/oracle positive-control diagnostic to test whether the AURORA-compatible "
        "allocation framework can express timing information when probability signals are deliberately informative."
    ),
    "important_caveat": (
        "The oracle probabilities are deliberately non-deployable because they use realized labels. "
        "They are used only to test diagnostic power, not to estimate investable performance."
    ),
    "inputs": {
        "label_file": str(label_path),
        "etf_return_source": str(etf_return_path),
        "evaluation_start": str(EVAL_START.date()),
        "evaluation_end": str(EVAL_END.date()),
        "n_evaluation_dates": int(len(eval_dates)),
    },
    "probability_design": {
        "formula": "p_tilde = (1-kappa)*p_prior + kappa*one_hot(y_t)",
        "kappa_values": KAPPA_VALUES,
        "alpha_20": ALPHA_20,
        "alpha_60": ALPHA_60,
        "prior20": prior20.tolist(),
        "prior60": prior60.tolist(),
        "shuffle_replications": N_SHUFFLE_REPLICATIONS,
        "base_random_seed": BASE_RANDOM_SEED,
    },
    "allocation_settings": {
        "lambda0": LAMBDA0,
        "gamma_U": GAMMA_U,
        "gamma_B": GAMMA_B,
        "rho": RHO,
        "mu_lookback": MU_LOOKBACK,
        "cov_lookback": COV_LOOKBACK,
        "mean_shrinkage": MEAN_SHRINKAGE,
        "momentum_weight": MOMENTUM_WEIGHT,
        "regime_tilt_strength": REGIME_TILT_STRENGTH,
        "cash_cap": CASH_CAP,
        "general_etf_cap": GENERAL_ETF_CAP,
        "etf_00881_cap": ETF_00881_CAP,
        "transaction_cost_bps": TRANSACTION_COST_BPS,
        "rebalance_convention": REBALANCE_CONVENTION,
        "scipy_available": SCIPY_AVAILABLE,
    },
    "outputs": {
        "run_root": str(RUN_ROOT),
        "table_S35": str(TABLE_RUN_DIR / "table_S35_oracle_positive_control_design.csv"),
        "table_S36": str(TABLE_RUN_DIR / "table_S36_oracle_positive_control_performance.csv"),
        "table_S36b": str(TABLE_RUN_DIR / "table_S36b_oracle_positive_control_ordered_compact.csv"),
        "table_S37": str(TABLE_RUN_DIR / "table_S37_oracle_ordered_vs_shuffled_comparison.csv"),
        "returns": str(RETURN_DIR / "oracle_positive_control_returns.parquet"),
        "weights": str(WEIGHT_DIR / "oracle_positive_control_weights.parquet"),
        "diagnostics": str(DIAG_DIR / "oracle_positive_control_diagnostics.csv"),
    },
    "educational_note": "Research diagnostics only; not personalized financial advice.",
}

validation_path = REPORT_RUN_DIR / "NOTEBOOK27_oracle_positive_control_validation_report.json"
global_validation_path = REPORT_DIR / f"NOTEBOOK27_oracle_positive_control_validation_report_{RUN_ID}.json"

save_json(validation_path, validation)
save_json(global_validation_path, validation)

manifest_df = make_file_manifest(RUN_ROOT)
manifest_path = REPORT_RUN_DIR / "NOTEBOOK27_file_manifest_SHA256.csv"
global_manifest_path = REPORT_DIR / f"NOTEBOOK27_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(global_manifest_path, index=False)

# ============================================================
# 9. Final summary
# ============================================================

print("\n" + "=" * 100)
print("NOTEBOOK 27 COMPLETE")
print("=" * 100)
print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", RUN_ROOT)
print("Table S35:", TABLE_RUN_DIR / "table_S35_oracle_positive_control_design.csv")
print("Table S36:", TABLE_RUN_DIR / "table_S36_oracle_positive_control_performance.csv")
print("Table S36b:", TABLE_RUN_DIR / "table_S36b_oracle_positive_control_ordered_compact.csv")
print("Table S37:", TABLE_RUN_DIR / "table_S37_oracle_ordered_vs_shuffled_comparison.csv")
print("Returns:", RETURN_DIR / "oracle_positive_control_returns.parquet")
print("Weights:", WEIGHT_DIR / "oracle_positive_control_weights.parquet")
print("Diagnostics:", DIAG_DIR / "oracle_positive_control_diagnostics.csv")
print("Validation report:", validation_path)
print("Manifest:", manifest_path)
print("=" * 100)

print("\nTable S35 preview:")
print(s35.to_string(index=False))

print("\nTable S36 ordered compact preview:")
print(s36_ordered_compact.round(6).to_string(index=False))

print("\nTable S37 preview:")
print(s37.round(6).to_string(index=False))

print("\nInterpretation helper:")
print(interpret_df.to_string(index=False) if not interpret_df.empty else "No interpretation rows.")

print("\nRecommended manuscript interpretation:")
print(
    "This oracle-probability positive control is deliberately non-deployable because it uses realized labels. "
    "Its purpose is to test whether the diagnostic framework can express timing information when probabilities "
    "are made informative. If ordered oracle probabilities increasingly outperform shuffled controls as kappa rises, "
    "the diagnostic framework has power to detect timing information. If cash-cap binding remains dominant even under "
    "strong oracle probabilities, the result supports the interpretation that allocation constraints suppress timing expression."
)
print("=" * 100)

Mounted at /content/drive
AURORA-TWETF Notebook 27: Oracle probability positive control
RUN_ID: 20260728_005644
RUN_ROOT: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/oracle_probability_positive_control/run_20260728_005644
Evaluation window: 2024-11-27 to 2026-03-25

Loading modeling labels and ETF returns
Label file: /content/drive/MyDrive/AURORA_TWETF/data/modeling/AURORA_TWETF_features_with_labels.parquet
Label shape: (1262, 2)
Label dates: 2021-01-06 to 2026-03-25
ETF return panel: /content/drive/MyDrive/AURORA_TWETF/data/panels/AURORA_etf_return_panel.parquet
ETF return shape: (1426, 4)
ETF return dates: 2021-01-01 to 2026-06-23
Aligned evaluation dates: 319 2024-11-27 to 2026-03-25
Prior 20d: {0: np.float64(0.0276), 1: np.float64(0.1676), 2: np.float64(0.4719), 3: np.float64(0.3054), 4: np.float64(0.0276)}
Prior 60d: {0: np.float64(0.0721), 1: np.float64(0.1676), 2: np.float64(0.263), 3: np.float64(0.3107), 4: np.float64(0.1866)}

Running oracle positive-control exper